# Travail Pratique 4

# <div class="alert alert-block alert-success">
<b>Informations générales </b>
</div>

**Professeur**: Stephan Robert

**Assistant(s)**: Léonard Truscello

**Contact**: stephan.o.robert@gmail.com, leonard.truscello@heig-vd.com ou via Teams

**Rendu**:

- Date: 
- Modalité: Travail individuel. Une fois complétées, copiez les réponses dans le ficher "**TP4_APV_S1-2026_Nom_Prenom.ipynb**" en remplaçant Nom et Prenom par les votres puis uploader votre fichier sur Cyberlearn.
- Note: Ce TP est noté sur 6, pour un poids de x%

**Étudiant**:

- Mouhamed Sakho

<div class="alert alert-block alert-success">

<b>But du TP </b>
</div>

Le but de ce travail pratique est d'implémenter une classification naïve bayésienne dans le cas continu.


<!---
<div class="alert alert-block alert-info">
<b>Rappel </b>
</div>
-->

<div class="alert alert-block alert-success">
<b>Table des matières </b>
</div>

1. Dataset
2. Naïve Bayes
3. Evaluation


Pour commencer, nous allons importer quelques packages qui vont nous être utile.

In [1]:
import numpy as np
import sklearn
from sklearn import datasets
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
# Si nécessaire ajoutez des librairies, sous-modules et/ou fonctions sous cette ligne
from scipy.stats import norm


## 1.  Dataset

Pour ce TP, nous allons travailler avec un dataset appelé 'Wine dataset'. Chaque observation correspond à des caractérisque de de vins. Les classes correspondent à trois types de vins différents.

In [2]:
# load dataset
wine_data = datasets.load_wine()

Comme toujours, il est important de comprendre les données avec lesquelles nous travaillons.

In [3]:
print(wine_data.DESCR)


.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

## 2. Naïve Bayes Classifier

Construisons maintenant un classificateur de type naïve bayes pour déterminer la classe. Cependant remarquer que dans notre cas, les features sont des valeurs continues (longueur et largeur des pétales et sépales).

Comme justifié dans la partie théorique de ce labo, notre modèle final sera de la forme 
$$
h_{\theta}(x) = argmax_{C_{j}} \log(p(C_{j})) + \sum_{i=1}^{n} \log(f_{\theta_{i, j}}(x_{i}))
$$
où $f_{\theta_{i, j}}$ est la fonction de densité de $X_{i}|C_{j}$

Ici, et comme il est très courant en pratique on va supposer que $X_{i}|C_{j} \sim \mathcal{N}(\mu, \sigma^{2})$ où $\mathcal{N}$ est une loi normale. Pour estimer $\mu$ et $\sigma$ et de manière très classique, on va utiliser la méthode du maximum de vraisemblance.

Rappelons que 
$$
\hat{\mu}_{mle} = \frac{1}{n} \sum_{i=1}^{n} X_{i}
$$
et que 
$$
\hat{\sigma}_{mle}^{2} = \frac{1}{n} \sum_{i=1}^{n} (X_{i} - \hat{\mu}_{mle})^{2}
$$

Nous allons maintenant implémenter le classifieur décrit ci-dessus en utilisant une classe appelée 'GaussianeNaiveBayes'.

In [4]:
class GaussianNaiveBayes:
    """
    An implementation of Naives Bayes classifier in continous case.
    """

    def __init__(self, data, target, feature_names=None):
        self.feature_names = feature_names
        self.classes = np.unique(target)
        # Estimations par maximum de vraisemblance, pour chaque classe C_j
        self.priors = np.array([np.mean(target == c) for c in self.classes])               # p(C_j)
        self.means = np.array([data[target == c].mean(axis=0) for c in self.classes])      # mu_{i,j}
        self.variances = np.array([data[target == c].var(axis=0) for c in self.classes])   # sigma^2_{i,j} (1/n)

    def classify(self, x):
        """Retourne argmax_j  log p(C_j) + sum_i log f_{i,j}(x_i)."""
        scores = [np.log(self.priors[j]) + np.sum(norm.logpdf(x, loc=self.means[j], scale=np.sqrt(self.variances[j])))
                  for j in range(len(self.classes))]
        return self.classes[np.argmax(scores)]


## 3. Evaluation

Nous allons maintenant construire et évaluer notre modèle. Pour commençons construisons notre training et test dataset.

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(wine_data.data, wine_data.target, test_size=0.3, random_state=2)

In [6]:
# Pour cet exercice, vous n'êtes pas obligé de construire votre modèle en utilisant les même input que moi.
# En particulier wine_data.feature_name n'est pas vraiment nécessaire
# TODO: Utilisez la ligne ci-dessous telle quelle ou adaptez là à votre modèle 
model = GaussianNaiveBayes(data=X_train, target=Y_train, feature_names=wine_data.feature_names)

Et une fois que notre modèle est construit, on peut essayer de calculer sa précision sur le test set.

In [7]:
# Ici aussi vous êtes très libre sur la manière d'utiliser votre modèle et des noms des méthodes que vous utilisez.
# TODO: Utilisez la ligne ci-dessous telle quelle ou adaptez là à votre modèle 
preds = [model.classify(x_test) for x_test in X_test]
print('Our model get an accuracy of {:.2f}% on the test dataset.'.format(100*((np.sum(preds == Y_test))/len(X_test))))

Our model get an accuracy of 96.30% on the test dataset.


En pratique, on a évidemmnent pas besoin d'implémenter nos modèles nous-mêmes. Par exemple, on peut utiliser GaussianNB de sklearn. 

On peut maintenant vérifier que notre classe GaussianNaiveBayes fournit les même résultats que GaussianNB.

In [8]:
official_model = GaussianNB()
official_model.fit(X_train, Y_train)
predictions = official_model.predict(X_test)
acc = np.round(accuracy_score(Y_test, predictions)*100, 2)
print('We obtain an accuracy of {}%'.format(acc))

We obtain an accuracy of 96.3%


<div class="alert alert-box alert-success">
<b>Question: </b>   
</div>

La variable random_state est fixé à 2 dans la construction du training et test datasets. Changez la valeur en random_state=None et relancez plusieurs fois l'intégralité du notebook. Qu'observez-vous et pourquoi obtient-on ces résultats ?   
   

Réponse: L'accuracy change d'une exécution à l'autre (pour les deux modèles, qui restent égaux entre eux). En la relançant une vingtaine de fois, nous avons obtenu des valeurs entre environ 94% et 100%, avec une moyenne vers 97%.

Avec `random_state=None`, `train_test_split` mélange les données différemment à chaque exécution : le training set et le test set ne contiennent donc pas les mêmes observations. Les paramètres estimés ($p(C_j)$, $\mu$, $\sigma^2$) changent un peu, et le test set (54 observations seulement) n'est plus le même. Comme une seule erreur vaut déjà $1/54 \approx 1.85\%$, l'accuracy varie de manière visible. Une seule accuracy mesurée sur un seul découpage n'est donc qu'une estimation de la vraie performance. Fixer `random_state` rend les résultats reproductibles, et on peut moyenner sur plusieurs découpages (validation croisée) pour avoir une estimation plus fiable.



<div class="alert alert-box alert-success">
<b>To do: </b>   
</div>

Après avoir répondu à cette question, refixez la variable random_state=2 dans le notebook que vous rendrez!